# Sprawozdanie L5 — SAC + HER (PandaReach-v3)


#### Zapis wag i hiperparametrów w `metadata.json`

Do skryptu treningowego dodano zapis wag (`policy.pt`) oraz plik `metadata.json`. Dzięki temu można pominąć pełny trening i wczytać wcześniej nauczoną politykę. Same tensory wag nie wystarczają: sieć musi mieć ten sam kształt, extractor, hiperparametry SAC/HER itd. Dlatego w metadanych zapisywana jest pełna **sygnatura** eksperymentu (`signature`): identyfikator środowiska, konfiguracja polityki, bufor HER, słownik argumentów przekazanych do `SAC`, liczba kroków treningu, ustawienia ewaluacji. Przy wczytywaniu sygnatura z pliku jest porównywana z sygnaturą wyliczoną z aktualnego kodu (`canonical_signature()` w `train.py`); przy rozbieżności program kończy działanie, żeby uniknąć cichego błędu.

W `metadata.json` znajdują się też pomocnicze pola, np. `saved_at_utc` (czas zapisu w UTC) oraz opcjonalnie `tensorboard_log_dir` (ścieżka do logów TensorBoard z tego runu, jeśli była podana przy zapisie).

![Zrzut ekranu z fragmentem metadanych lub IDE pokazującym strukturę katalogu runu, pole `signature` oraz plik wag](<Screenshot 2026-05-30 202306.png>)


## Implementacja — zapis i wczytywanie runu

Po zakończeniu treningu katalog runu (np. `weights/2026-05-24_16-53/`) otrzymuje `metadata.json` oraz `policy.pt`. Funkcja `load_sac_from_run_dir` czyta metadane, wywołuje `_assert_signature_matches`, buduje środowisko i algorytm przez `instantiate_sac_from_signature`, a następnie `algo.load(...)`. Z linii poleceń tryb tylko-wczytaj uruchamia się przez `python train.py --load ścieżka/do/katalogu_runu` (render testowy bez treningu).

Poniżej fragmenty z [`train.py`](train.py) (skrócone komentarzami `...` tam, gdzie pełna treść jest oczywista z kontekstu).


```python
# train.py — sygnatura eksperymentu (zgodność zapis ↔ aktualny kod)
def canonical_signature(sac_branch: dict[str, Any] | None = None) -> dict[str, Any]:
    sac = dict(SAC_KWARGS) if sac_branch is None else dict(sac_branch)
    return {
        "format_version": 1,
        "env_id": DEFAULT_ENV_ID,
        "policy": {
            "hidden_sizes": list(POLICY_HIDDEN),
            "extractor": POLICY_EXTRACTOR,
        },
        "her": {
            "n_sampled_goal": HER_N_SAMPLED_GOAL,
            "goal_selection_strategy": HER_STRATEGY,
            "replay_buffer_size_train": REPLAY_BUFFER_SIZE_TRAIN,
            "replay_buffer_size_load": REPLAY_BUFFER_SIZE_LOAD,
        },
        "sac": sac,
        "train_loop": {"n_steps": TRAIN_N_STEPS, "log_interval": TRAIN_LOG_INTERVAL},
        "eval_render": {"n_episodes": EVAL_N_EPISODES, "sleep": EVAL_RENDER_SLEEP},
    }

def save_training_run(algo: SAC, run_dir: Path, *, sac_kwargs=None, tensorboard_log_dir=None) -> None:
    run_dir.mkdir(parents=True, exist_ok=True)
    effective_sac = dict(SAC_KWARGS) if sac_kwargs is None else dict(sac_kwargs)
    sig = canonical_signature(sac_branch=effective_sac)
    doc = {
        "signature": sig,
        "weights_file": WEIGHTS_FILENAME,
        "saved_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    tb_meta = _tensorboard_dir_for_metadata(tensorboard_log_dir)
    if tb_meta is not None:
        doc["tensorboard_log_dir"] = tb_meta
    meta_path = run_dir / METADATA_FILENAME
    with meta_path.open("w", encoding="utf-8") as f:
        json.dump(doc, f, indent=2, sort_keys=True)
        f.write("\n")
    algo.save(str(run_dir / WEIGHTS_FILENAME))

def load_sac_from_run_dir(run_dir: str | Path) -> Tuple[gym.Env, SAC, dict[str, Any]]:
    run_path = Path(run_dir).expanduser().resolve()
    meta_path = run_path / METADATA_FILENAME
    with meta_path.open(encoding="utf-8") as f:
        doc = json.load(f)
    _assert_signature_matches(doc["signature"])
    weights_path = run_path / doc.get("weights_file", WEIGHTS_FILENAME)
    device = resolve_device()
    env, algo = instantiate_sac_from_signature(doc["signature"], device=device)
    algo.load(str(weights_path))
    return env, algo, doc

# train.py — CLI (fragment)
parser.add_argument(
    "--load", type=Path, metavar="RUN_DIR",
    help="Katalog runu (metadata.json + policy.pt)",
)
args = parser.parse_args()
if args.load is None:
    main_train()
else:
    run_eval_render_from_run_dir(args.load.expanduser().resolve())
```


## Implementacja — automatyczne $\alpha$ w SAC

W wersji szkieletowej z katalogu `OG_source_code` w `train.py` używane jest stałe `alpha=0.05`, a w `asdf/algos.py` sekcja „auto” ma niedokończone TODO (`compute_loss_alpha` zwraca zera, brak stabilnego `alpha_optimizer` w `update`). W tej pracy włączono uczenie temperatury entropii zgodnie z sekcją *Automating Entropy Adjustment* w pracy [SAC — Applications](https://arxiv.org/abs/1812.05905): zamiast stałego $\alpha$ optymalizowany jest parametr $\log\alpha$, a $\alpha = \exp(\log\alpha)$. Docelowa entropia przy `target_entropy="auto"` jest ustawiana na $-\dim(\mathcal{A})$ (iloczyn wymiarów przestrzeni akcji), co dla ciągłych akcji typu Panda odpowiada skalarowi używanemu w stracie dla $\log\alpha$.

Wartość $\alpha$ wpływa na backup Bellmana (kary entropii przy estymacji $Q$) oraz na stratę polityki; przy uczeniu $\log\alpha$ współczynnik w stratach $Q$ i $\pi$ jest **odłączany od grafu** względem aktualizacji $\alpha$, żeby uniknąć niepożądanego przepływu gradientu (jak w dyskusji [softlearning#60](https://github.com/rail-berkeley/softlearning/issues/60)).

W [`train.py`](train.py) w `SAC_KWARGS` ustawiono `alpha="auto"` oraz `target_entropy="auto"`.

**Kontrast z `OG_source_code`.** Szkielet pozostawiał `alpha` jako stałą tensorową lub niekompletny blok `auto`; poniższy kod z [`asdf/algos.py`](asdf/algos.py) domyka inicjalizację, straty, krok optymalizatora oraz zapis/wczyt checkpointu dla nauczalnego $\log\alpha$.


```python
# asdf/algos.py — inicjalizacja auto-α (fragment __init__)
if isinstance(alpha, str) and alpha.startswith("auto"):
    init_value = 1.0
    if "_" in alpha:
        init_value = float(alpha.split("_")[1])
    self.log_alpha = nn.Parameter(
        torch.tensor(np.log(init_value), dtype=torch.float32, device=alpha_device)
    )
    self.alpha_optimizer = Adam([self.log_alpha], lr=lr)
    with torch.no_grad():
        self.alpha = self.log_alpha.exp().detach()
else:
    self.alpha = torch.tensor(float(alpha), dtype=torch.float32, device=alpha_device)
    self.alpha_optimizer = None
    self.log_alpha = None

def _entropy_coefficient(self, detach: bool) -> torch.Tensor:
    if self.alpha_optimizer is None:
        ent = self.alpha
    else:
        ent = self.log_alpha.exp()
    return ent.detach() if detach else ent

# Bellman backup: ent_coef odłączony przy Q
ent_coef = self._entropy_coefficient(detach=True)
backup = r + self.gamma * (1 - ter) * (q_pi_targ - ent_coef * logp_a2)

# Strata polityki
loss_pi = (self._entropy_coefficient(detach=True) * logp_pi - q_pi).mean()

def compute_loss_alpha(self, logp_pi):
    if self.alpha_optimizer is None:
        return torch.tensor(0.0, device=logp_pi.device, dtype=logp_pi.dtype), self.alpha
    alpha_loss = -(self.log_alpha * (logp_pi + self.target_entropy).detach()).mean()
    return alpha_loss, self.log_alpha.exp()

# Krok optymalizatora α w update()
if self.alpha_optimizer is not None:
    alpha_loss, _ = self.compute_loss_alpha(logp_pi)
    self.alpha_optimizer.zero_grad()
    alpha_loss.backward()
    self.alpha_optimizer.step()
    with torch.no_grad():
        self.alpha = self.log_alpha.exp().detach()

# save / load — log_alpha + alpha_optimizer gdy auto
def save(self, path: str):
    payload = {
        "policy_state_dict": self.policy.state_dict(),
        "pi_optimizer_state_dict": self.pi_optimizer.state_dict(),
        "q_optimizer_state_dict": self.q_optimizer.state_dict(),
    }
    if self.alpha_optimizer is not None:
        payload["log_alpha"] = self.log_alpha.detach().cpu()
        payload["alpha_optimizer_state_dict"] = self.alpha_optimizer.state_dict()
    else:
        payload["alpha"] = self.alpha
    torch.save(payload, path)

def load(self, path: str):
    checkpoint = torch.load(path, map_location=next(self.policy.parameters()).device)
    dev = next(self.policy.parameters()).device
    # ... load_state_dict dla policy, pi_optimizer, q_optimizer ...
    if "log_alpha" in checkpoint and self.alpha_optimizer is not None:
        self.log_alpha.data.copy_(checkpoint["log_alpha"].to(dtype=self.log_alpha.dtype, device=dev))
        self.alpha_optimizer.load_state_dict(checkpoint["alpha_optimizer_state_dict"])
        with torch.no_grad():
            self.alpha = self.log_alpha.exp().detach()
    else:
        self.alpha = checkpoint["alpha"].to(device=dev)
```


## Implementacja — HER (`HerReplayBuffer`)

**Hindsight Experience Replay** relabeluje cele po zakończeniu epizodu: każdy krok zapisywany jest z oryginalnym `desired_goal` oraz `n_sampled_goal` razy z nowym celem wylosowanym według strategii (`future` — cele z przyszłych osiągniętych stanów w tym samym epizodzie). Nagroda przeliczana jest przez `env.unwrapped.compute_reward`. W `OG_source_code/asdf/buffers.py` klasa `HerReplayBuffer` zawierała jedynie TODO; poniżej rdzeń implementacji z [`asdf/buffers.py`](asdf/buffers.py).

W [`train.py`](train.py) użyto `HER_STRATEGY = "future"` oraz `HER_N_SAMPLED_GOAL = 3`.


```python
# asdf/buffers.py — zapis epizodyczny i flush HER przy końcu epizodu
# (na górze pliku: from copy import deepcopy)
def end_episode(self) -> None:
    ep = self._episode_transitions
    if not ep:
        return
    env_compute = self.env.unwrapped
    for t_idx, tr in enumerate(ep):
        self._store_one_transition(
            tr["observation"], tr["action"], tr["reward"],
            tr["next_observation"], tr["terminated"], tr["truncated"], tr["info"],
        )
        for _ in range(self.n_sampled_goal):
            new_goal = self._sample_goal(ep, t_idx)
            o = self._copy_obs(tr["observation"])
            o2 = self._copy_obs(tr["next_observation"])
            o["desired_goal"] = np.asarray(new_goal, dtype=np.float32).copy()
            o2["desired_goal"] = np.asarray(new_goal, dtype=np.float32).copy()
            info = deepcopy(tr["info"])
            r = float(
                env_compute.compute_reward(
                    o2["achieved_goal"], o2["desired_goal"], info
                )
            )
            self._store_one_transition(
                o, tr["action"], r, o2, tr["terminated"], tr["truncated"], info,
            )

def _sample_goal(self, episode, transition_idx: int) -> NDArray:
    if self.selection_strategy == "future":
        candidates = [
            np.asarray(episode[j]["next_observation"]["achieved_goal"], dtype=np.float32).copy()
            for j in range(transition_idx + 1, len(episode))
        ]
        if not candidates:
            return self._sample_goal_episode(episode)
        pick = int(self._rng.integers(0, len(candidates)))
        return np.asarray(candidates[pick], dtype=np.float32).copy()
    # ... final, episode ...
```


## Wyniki — krzywe z TensorBoard

Poniższe wykresy pochodzą z loggera skalarnego w pętli treningowej (`asdf/algos.py`: `loss_q`, `loss_pi`, `loss_alpha`, `alpha`; oraz `ep_return`, `ep_length`, `test_ep_return`, `test_ep_length`). Oś pozioma to zwykle krok środowiska $t$. Konkretne wartości odczytaj z osi na załączonych PNG — opis odnosi się do **znaczenia** metryki.

### `first.png`

![TensorBoard — pierwszy zrzut / dashboard](first.png)

Zrzut zbiorczy z TensorBoard (pierwszy ekran lub układ kilku kart). Służy do szybkiego porównania trendów metryk treningu i testu na jednym widoku.

### `alpha.png`

![Skalar alpha](<alpha.png>)

Ewolucja współczynnika temperatury entropii $\alpha$ (po uczeniu: $\exp(\log\alpha)$). Oczekiwany obraz to adaptacja poziomu eksploracji: $\alpha$ może maleć, gdy polityka już „niesie” wystarczającą entropię względem docelowej, lub oscylować wokół wartości zgodnej z `target_entropy`.

### `loss_alpha.png`

![Strata alpha](<loss_alpha.png>)

Wartość `loss_alpha` to strata dla parametru $\log\alpha$. Gdy średnia entropia polityki zbliża się do $-\dim(A)$, człon $(\log\pi + H_{\text{targ}})$ dąży do zera i krzywa często stabilizuje się przy małych wartościach.

### `loss_pi.png`

![Strata polityki](<loss_pi.png>)

Strata aktora SAC: średnia z $\alpha \log \pi(a|s) - Q(s,a)$ (z aktualnej polityki stochastycznej). Spadek nie musi być monotoniczny — współistnieje z aktualizacją sieci $Q$ i ze zmianą $\alpha$.

### `loss_q.png`

![Strata Q](<loss_q.png>)

Średnia strata obu krytyków względem kopii Bellmana (`MSE` między $Q_1,Q_2$ a targetem z sieci docelowych i polityki). Duże skoki na początku są typowe; później krzywa zwykle jest gładsza.

### `loss_q_2.png`

![Strata Q — drugi widok](<loss_q_2.png>)

Drugi zrzut tej samej metryki `loss_q` (inny zakres osi, inny run lub inna skala czasu). Porównaj z `loss_q.png`, jeśli eksperyment był powtarzany z innymi hiperparametrami.

### `ep_length.png`

![Długość epizodu treningowego](<ep_length.png>)

Długość epizodów podczas **zbierania** doświadczeń (stochastyczna polityka + random start). W zadaniach reach krótsze epizody często korelują z szybszym osiągnięciem celu; na początku treningu długości bywają bliskie `max_episode_len`.

### `test_ep_length.png`

![Długość epizodu testowego](<test_ep_length.png>)

Średnia długość epizodu przy **deterministycznej** akcji (`policy.act(..., deterministic=True)`), co `log_interval` kroków, uśredniona po `n_test_episodes`. Spadek wskazuje, że agent szybciej kończy epizod testowy.

### `test_ep_return.png`

![Zwrot testowy](<test_ep_return.png>)

Średni zwrot epizodu w tej samej procedurze testowej. Rosnący trend wskazuje na poprawę polityki na PandaReach (nagrody są ujemne w typowej konfiguracji — wtedy „lepsze” to mniej ujemne wartości bliżej zera).

---

### Wnioski

